In [ ]:
# --- Importing Libraries
import sys
print("Python Executable:", sys.executable)

import pandas as pd # type: ignore
import numpy as np
np.int = int # 😠 come on pygam
import matplotlib.pyplot as plt # type: ignore
import seaborn as sns # type: ignore
import statsmodels.api as sm # type: ignore
import statsmodels.formula.api as smf # type: ignore
import matplotlib.pyplot as plt # type: ignore
from stargazer.stargazer import Stargazer # type: ignore
from IPython.display import HTML, display # type: ignore
from sklearn.preprocessing import MinMaxScaler # type: ignore
import re
import matplotlib.ticker as ticker
from matplotlib.ticker import FuncFormatter
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNetCV
from sklearn.pipeline import make_pipeline
from sklearn.metrics import r2_score, root_mean_squared_error
from sklearn.utils import resample
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.inspection import PartialDependenceDisplay
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.utils.validation import check_is_fitted
from sklearn.gaussian_process.kernels import Matern
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from pygam import LinearGAM, s
import pymc as pm
#Hacking together version deprecationsin pygam
import scipy.sparse
scipy.sparse.csr.csr_matrix.A = property(lambda self: self.toarray())


plt.style.use('dark_background')


# Create a font object
from matplotlib import font_manager as fm
poiret = fm.FontProperties(fname="C:/Users/ben.pharris/Downloads/Poiret_One/Montserrat-SemiBold.ttf",size=None)
from matplotlib import font_manager
font_manager.fontManager.addfont("C:/Users/ben.pharris/Downloads/Poiret_One/Montserrat-SemiBold.ttf")

plt.rcParams['font.family'] = poiret.get_name()

In [ ]:
# --- Read in Data, clean columns, convert date to datetime
file_path = "../FunctionalEstimation/data/processed/mkt_2h_2024.csv"
df = pd.read_csv(file_path)
del file_path

df['Date'] = pd.to_datetime(df['Date'])
del df['Indirect Traffic']

df.info()

In [ ]:
test_data = pd.read_csv('C:/Users/ben.pharris/Downloads/Q12025.csv')
pd.to_datetime(test_data['date'])

test_data.head()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.preprocessing import StandardScaler

# Filter Q4 2024 from df
q4 = df[df['date'].between('2024-10-01', '2024-12-31')].copy()
q1 = test_data.copy()  # Assuming this is already Q1 2025

# Align column naming
q4['GA'] = q4['sales']
q1['GA'] = q1['ga']

q4['date'] = pd.to_datetime(q4['date'])
q1['date'] = pd.to_datetime(q1['date'])

# Combine for joint scaling
q4['quarter'] = 'Q4 2024'
q1['quarter'] = 'Q1 2025'
combined = pd.concat([q4, q1])

# Standardize GA and total_spend together
scaler = StandardScaler()
combined[['GA_std', 'Spend_std']] = scaler.fit_transform(combined[['GA', 'total_spend']])

fig, axes = plt.subplots(1, 2, figsize=(18,8), sharey=True, facecolor='black')

# … your two‐panel plotting loop …
for ax, (label, group) in zip(axes, combined.groupby('quarter')):
    ax.plot(group['date'], group['GA_std'], label='GA', linewidth=2, color='firebrick')
    ax.plot(group['date'], group['Spend_std'], label='Working Media Spend', linewidth=2, color='#839fae')
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
    ax.set_title(label, color='white')
    ax.set_xlabel('Month', color='white')
    ax.grid(False)
    ax.tick_params(axis='x', colors='white')
    ax.tick_params(axis='y', colors='white')

# 1) Remove ticks & labels on the Q4 subplot (axes[0])
axes[0].tick_params(
    axis='both',       # both axes
    which='both',      # major and minor
    bottom=False,      # turn off bottom ticks
    top=False,         # turn off top ticks
    left=True,        # turn off left ticks
    right=False,       # turn off right ticks
    labelbottom=True, # turn off x-labels
    labelleft=True    # turn off y-labels
)

axes[0].set_facecolor('black')

# 2) Figure‐level legend under the title
#    grab the handles & labels from either axis
handles, labels = axes[1].get_legend_handles_labels()
fig.legend(
    handles, labels,
    loc='upper center',
    ncol=2,
    frameon=False,
    fontsize=9,
    bbox_to_anchor=(0.5, 0.95),  # x=0.5 center, y=0.95 just below suptitle
    borderaxespad=0.  # no extra padding
)

# 3) Suptitle and layout adjustments
plt.suptitle('GA vs. Total Marketing Spending', color='white', fontsize=16)
plt.subplots_adjust(top=0.88)  # make space for the legend/suptitle
plt.tight_layout(rect=[0, 0.01, 1, 0.88])  # leave top free

plt.show()

In [ ]:
# Define sub-channels
channels = ['email', 'local', 'paid_search', 'paid_shopping', 'paid_social', 'program']

# Ensure total_spend exists
test_data['total_spend'] = test_data[channels].sum(axis=1)

# Calculate proportional spend per channel
for ch in channels:
    test_data[f'{ch}_share'] = test_data[ch] / test_data['total_spend']

# Stackplot data: x = dates, y = list of share arrays (in order)
x = pd.to_datetime(test_data['date'])
y = [test_data[f'{ch}_share'] for ch in channels]

# Plot
plt.figure(figsize=(14, 6))
plt.stackplot(x, y, labels=channels, alpha=0.9)

plt.title('Channel Share of Total Spend Over Q1 2025')
plt.xlabel('Date')
plt.ylabel('Share of Spend')
plt.legend(loc='upper left', ncol=3)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import test

# — assume test_data already has all the *_z_frac, ga_total_z, cpga_smooth, plus spend_total & ga_total —


test_data['cpga']  = test_data['total_spend'] / test_data['ga']
test_data['ga_total'] = test_data['ga']
test_data['spend_total'] = test_data['total_spend']

spend_ch     = ['email','local','paid_search','paid_shopping','paid_social','program']
spend_colors = ["#c9d2db", "#b3c2ce", "#9bb2c0", "#839fae", "#6d8c9b", "#587886"]

x = pd.to_datetime(test_data['date'])
bar_width = pd.Timedelta(days=0.85)

for col in ['spend_total','ga_total',  'cpga']:
    μ, σ = test_data[col].mean(), test_data[col].std()
    test_data[col + '_z'] = (test_data[col] - μ) / σ



# 4 rows, share x-axis; give bottom plot most room
fig, (ax0, ax1, ax2, ax3) = plt.subplots(
    4, 1,
    figsize=(13, 12),
    sharex=True,
    facecolor='black',
    gridspec_kw={'height_ratios': [1, 1, 1, 3]}
)

# black background on all
for ax in (ax0, ax1, ax2, ax3):
    ax.set_facecolor('black')

# === Panel 1: Total actual GAs ===
ax0.bar(x, test_data['ga'], color='tomato', width=bar_width, edgecolor='none')
ax0.set_ylabel('Total GAs', color='white', rotation=0, labelpad=40)
for spine in ['top','right','bottom']:
    ax0.spines[spine].set_visible(False)
ax0.tick_params(axis='x', which='both', bottom=False, labelbottom=False)
ax0.tick_params(axis='y', colors='white')

# === Panel 2: Total actual Spend ===
ax1.bar(x, test_data['total_spend'], width=bar_width, color=spend_colors[3], edgecolor='none')
ax1.set_ylabel('Media Spend', color='white', rotation=0, labelpad=40)
for spine in ['top','right','bottom']:
    ax1.spines[spine].set_visible(False)
ax1.tick_params(axis='x', which='both', bottom=False, labelbottom=False)
ax1.tick_params(axis='y', colors='white')

# === Panel 3: CPGA bar chart ===
ax2.bar(
    x,
    test_data['cpga'],
    width=bar_width,
    color='goldenrod'
)
ax2.set_ylabel('CPGA', color='white', rotation=0, labelpad=40)
for spine in ['top','right','bottom']:
    ax2.spines[spine].set_visible(False)
ax2.tick_params(axis='x', which='both', bottom=False, labelbottom=False)
ax2.tick_params(axis='y', colors='white')

months = pd.date_range(
    start=test_data['date'].min(),
    end=test_data['date'].max(),
    freq='MS'    # Month Start
)


# === Panel 4: Stacked spend, GA bars & CPGA ribbon ===
# 4.1) Stacked spend

cols = ['spend_total_z', 'ga_total_z', 'cpga_z']
test_data[cols] = (
    test_data[cols]
      .rolling(window=7, min_periods=1, center=True)
      .mean()
)

# 4.2) Outline of total spend_z
ax3.plot(
    x,
    test_data['spend_total_z'],
    color='#839fae',
    linewidth=2,
    zorder=2
)

ax3.fill_between(
    x,
    test_data['spend_total_z'],
    color='#b3c2ce',
    alpha=0.2,
    zorder=0
)

# 4.3) GA total bars (narrower)
ax3.plot(
    x,
    test_data['ga_total_z'],
    color='firebrick',
    linewidth=2,
    zorder=2
)

ax3.fill_between(
    x,
    test_data['ga_total_z'],
    color='tomato',
    alpha=0.2,
    zorder=0
)

# 4.4) CPGA smooth line + fill
ax3.plot(
    x,
    test_data['cpga_z'],
    color='gold',
    linewidth=1,
    zorder=3
)
ax3.fill_between(
    x,
    test_data['cpga_z'],
    color='gold',
    alpha=0.2,
    zorder=0
)

# 4.5) Styling
ax3.axhline(0, color='white', linewidth=1)
ax3.xaxis.set_major_locator(mdates.MonthLocator())
ax3.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
ax3.set_xlim(pd.Timestamp('2025-01-01'), pd.Timestamp('2025-03-31'))

# hide y-axis on bottom
ax3.set_ylabel('Performance\n+/-', rotation=0, labelpad=40)
ax3.tick_params(axis='y', left=True, labelleft=True, colors='white')
for spine in ['top','right']:
    ax3.spines[spine].set_visible(False)
ax3.spines['bottom'].set_visible(True)
ax3.tick_params(axis='x', colors='white', rotation=0)

# === Legend on bottom panel ===
# draw a thin gray vline on ax0, ax1, ax2 for each month
for ax in (ax0, ax1, ax2,ax3):
    for m in months:
        ax.axvline(
            m,
            color='white',
            linestyle='-',
            linewidth=1,
            alpha=0.6,
            zorder=0
        )

# create one patch for "Marketing Spend"
marketing_line = Line2D([0],[0], color='#839fae', linewidth=2, label='Media Spend')
ga_line = Line2D([0],[0], color='firebrick', linewidth=2, label='GA')
cpga_line = Line2D([0],[0], color='gold', linewidth=2, label='CPGA')

fig.legend(
    handles=[marketing_line, ga_line, cpga_line],
    loc='upper center',
     borderpad=0.5, 
    frameon=False,
    fontsize=12,
    ncol=3,
    bbox_to_anchor=(0.54,1)
)
plt.subplots_adjust(top=0.85)
plt.tight_layout(h_pad=1.0)
plt.show()